# linspace-out-param — worked example 1: Fill a noise schedule buffer with linspace out=

> Worked example from [Delta Drills](https://delta-drills.vercel.app). Atom: `linspace-out-param`.

**This is a worked example — read it, run each cell, and follow the reasoning.** It's study material, so there's nothing to submit here. Delta Drills hands you a hands-on version to complete yourself as you get comfortable with the idea.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)

## Concept

_First time on this topic? Run the **Setup** cell above and skim it: every class and helper mentioned below is defined there. You don't need to have done any other drill first._

`torch.linspace(start, end, N, out=buf)` writes the ramp directly into an existing buffer and returns that same buffer object (aliased), avoiding a fresh allocation each call. This is the zero-alloc pattern for rebuilding a schedule inside a hot loop.

## Worked solution

We rebuild a diffusion-style beta schedule into one reusable buffer.

1. **Pre-allocate once.** The buffer `buf` is created before the loop; every iteration must write into it, never allocate a new tensor.
2. **Fill in place.** For each `beta_end`, `torch.linspace(beta_start, beta_end, N, out=buf)` overwrites all N entries of `buf` with the ramp from `beta_start` to `beta_end`.
3. **Aliasing.** The call returns `buf` itself, so its `data_ptr()` never changes. We record `buf.sum().item()` right after each fill to prove the correct values lived there at that moment.
4. **Why out=.** In a sampler you rebuild the schedule many times; `out=` keeps memory traffic flat instead of churning a new tensor per step.

The demo fills the buffer for three different endpoints and prints the running sums plus a confirmation that the pointer stayed constant.

In [ ]:
import torch as t

t.manual_seed(0)
N = 6
buf = t.zeros(N)
ptr0 = buf.data_ptr()

def fill_schedule(buf, beta_start, beta_ends):
    sums = []
    for beta_end in beta_ends:
        t.linspace(beta_start, beta_end, buf.numel(), out=buf)
        sums.append(buf.sum().item())
    return sums

sums = fill_schedule(buf, 0.0001, [0.02, 0.05, 0.1])
print('per-fill sums:', [round(s, 4) for s in sums])
print('pointer stable:', buf.data_ptr() == ptr0)
print('last buffer:', buf.tolist())